# 2 · Building the supersedence rule

The obvious rule is *"a newer version exists, so this one is superseded"*. The
obvious rule is wrong in a way that only shows up if you measure it over the
whole population instead of over the rows you are trying to catch.

This notebook builds the rule the way it actually got built: propose, measure
precision **and** recall, discard, repeat.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)

from src.synthetic import generate
from src.rules import frame_from_names, classify, is_superseded, explain
from src.agreement import score_rule, base_rate

findings = frame_from_names(generate())
print("base rate:", base_rate(findings, "SUPERSEDED"))

base rate: 0.1135


## Attempt 1 — any older version at all

Compare the full dotted version and flag anything below the highest one seen for
that product. It reads correctly in English, which is exactly the problem.

In [ ]:
def version_key(series):
    # Zero-pad each segment so 150.0.4078.83 sorts below 150.0.4078.105
    return (series.fillna("")
            .str.split(".")
            .apply(lambda parts: ".".join(p.zfill(6) for p in parts if p)))

versions = version_key(findings["definition_name"].str.extract(
    r"<\s*=?\s*(\d+(?:\.[0-9A-Za-z]+)*)", expand=False))
newest_version = versions.groupby(findings["product"]).transform("max")
naive = findings["product"].notna() & versions.ne("") & versions.lt(newest_version)

print("full-version comparison:", score_rule(naive, findings, "SUPERSEDED"))

full-version comparison: {'fired': 248, 'hits': 136, 'other_category': 75, 'unlabelled': 37, 'precision': 0.5484, 'recall': 0.3293}


## Attempt 2 — only a newer *branch* counts

`151.0.7922.71` and `151.0.7922.108` are two patches of the same branch. The
vendor did not replace the plugin, it revised it. Only a change of major version
means the old plugin has been superseded.

The difference between the two rules is a single `.str.split(".")` — and a large
difference in precision.

In [ ]:
print("major comparison:      ", score_rule(is_superseded(findings), findings, "SUPERSEDED"))
print()
print("rows the naive rule adds:", int((naive & ~is_superseded(findings)).sum()))
extra = findings[naive & ~is_superseded(findings)]
print(extra["analyst_category"].value_counts(dropna=False).to_string())

major comparison:       {'fired': 158, 'hits': 130, 'other_category': 0, 'unlabelled': 28, 'precision': 0.8228, 'recall': 0.3148}

rows the naive rule adds: 90
analyst_category
PATCH          68
<NA>            9
NOT_SCANNED     7
SUPERSEDED      6


Same recall, better precision. The extra rows the naive rule picks up are
same-branch patches the analyst does not consider superseded at all.

## The precedence problem

A finding on a machine that has not been scanned recently has *two* things true
about it at once: the plugin was replaced, **and** the scan is stale. Which one
wins is not a detail — it decides where the row is reported and who acts on it.

The engine models these as two levels: a **state** (is this actionable at all?)
resolves first, and only genuinely active findings get a **workflow**. The
`supersedence_first` switch keeps the losing design runnable so the decision can
be measured instead of argued.

In [ ]:
from src.agreement import agreement, totals

for supersedence_first in (False, True):
    result = classify(findings, supersedence_first=supersedence_first)
    report = agreement(result)
    row = report.set_index("category").loc["SUPERSEDED"]
    print(f"supersedence_first={supersedence_first!s:5}  "
          f"SUPERSEDED agreed={int(row['agreed']):4}  "
          f"missed={int(row['missed']):4}  "
          f"overall agreed={totals(report)['agreed']:5}  "
          f"similarity={totals(report)['similarity']}")

supersedence_first=False  SUPERSEDED agreed= 104  missed= 309  overall agreed= 2713  similarity=0.672
supersedence_first=True   SUPERSEDED agreed= 130  missed= 283  overall agreed= 2739  similarity=0.6829


Resolving the scan state first silently swallows findings that belong to the
supersedence state — and, because they land in a bucket that looks healthy, the
loss is invisible in any per-category total.

## Explaining a single row

A rule engine that cannot answer *"why this category?"* does not survive contact
with the analyst whose work it is reproducing. Precedence is stored as data, so
the answer is an index lookup rather than an archaeology exercise.

In [ ]:
result = classify(findings)
for category in ("SUPERSEDED", "NOT_SCANNED", "CERTIFICATE", "PATCH"):
    index = result.index[result["category"].eq(category)][0]
    print(f"{result.loc[index, 'definition_name'][:58]:60} -> {explain(findings, index)}")

Contoso Browser (Chromium) < 150.0.4078.83 Multiple Vulner   -> SUPERSEDED: state resolved before any workflow (product 'contoso browser (chromium)' is on branch 151 in this cut, this finding is branch 150)
Contoso Browser (Chromium) < 151.0.4129.59 Multiple Vulner   -> NOT_SCANNED: state resolved before any workflow (last authenticated scan 15 days ago, threshold 10)
SSL Certificate Cannot Be Trusted                            -> CERTIFICATE: rule 4 of 6 in the cascade — certificate hygiene
Contoso Browser (Chromium) < 151.0.4129.59 Multiple Vulner   -> PATCH: rule 6 of 6 in the cascade — ordinary patch cycle
